In [13]:
import pandas as pd
import geopandas as gpd
import joblib
import numpy as np
import warnings

In [14]:
def muat_data_dan_model(file_model, file_input_cuaca):
    """Memuat model dan data input cuaca."""
    print("[1/5] Memuat model dan data input cuaca...")
    try:
        model = joblib.load(file_model)
        df_cuaca = pd.read_csv(file_input_cuaca)
        df_cuaca.rename(columns={
            'rata_rata_humi': 'humidity',
            'Curah_Hujan_Rata': 'rainfall',
            'rata_rata_temp': 'temperature'
        }, inplace=True, errors='ignore')
        return model, df_cuaca
    except FileNotFoundError as e:
        print(f"[ERROR] Gagal memuat file: {e}")
        return None, None

In [ ]:
def get_kategori(prob):
    if prob >= 0.75:
        return "sangat ideal"
    elif prob >= 0.65:
        return "mendekati ideal"
    else:
        return "cukup ideal"

In [1]:
def lakukan_prediksi_awal(model, df_cuaca):
    """Melakukan prediksi awal hanya berdasarkan cuaca."""
    print("[2/5] Melakukan prediksi awal berdasarkan cuaca...")
    # Pastikan semua fitur yang dibutuhkan ada
    fitur_wajib = ['temperature', 'humidity', 'rainfall']
    if not all(fitur in df_cuaca.columns for fitur in fitur_wajib):
        print(f"[ERROR] Data input cuaca tidak memiliki semua kolom yang dibutuhkan: {fitur_wajib}")
        return None
        
    fitur_cuaca = df_cuaca[fitur_wajib]
    prediksi_proba = model.predict_proba(fitur_cuaca)
    kelas_model = model.classes_
    
    hasil_awal = []
    for probas in prediksi_proba:
        top_indices = probas.argsort()[-3:][::-1]
        top_crops = [
            {
                'nama': kelas_model[i], 
                'skor': f"{probas[i] * 100:.0f}%",
                'kategori': get_kategori(probas[i])
            } 
            for i in top_indices
        ]
        hasil_awal.append(top_crops)
        
    df_cuaca['prediksi_awal'] = hasil_awal
    return df_cuaca

SyntaxError: invalid syntax (1015038277.py, line 21)

In [16]:
def gabung_dengan_data_lahan(df_cuaca, file_peta_lahan):
    """
    Menggabungkan data cuaca dengan data peta lahan lokal (Spatial Join).
    """
    print(f"[3/5] Memuat peta tata guna lahan dari file lokal '{file_peta_lahan}'...")
    try:
        gdf_lahan = gpd.read_file(file_peta_lahan)
    except Exception as e:
        print(f"[ERROR] Gagal membaca file GeoJSON/Shapefile: {e}")
        return None

    # Mengubah dataframe cuaca menjadi GeoDataFrame
    # Pastikan nama kolom lat/lon di file cuaca Anda sudah benar ('LON', 'LAT')
    gdf_cuaca = gpd.GeoDataFrame(
        df_cuaca, 
        geometry=gpd.points_from_xy(df_cuaca.LON, df_cuaca.LAT),
        crs="EPSG:4326"
    )
    
    # Pastikan kedua layer memiliki sistem koordinat yang sama
    if gdf_cuaca.crs != gdf_lahan.crs:
        print("      Menyamakan sistem koordinat (CRS)...")
        gdf_cuaca = gdf_cuaca.to_crs(gdf_lahan.crs)
    
    print("[4/5] Melakukan Spatial Join untuk identifikasi jenis lahan...")
    # Lakukan spatial join (menumpuk titik di atas poligon)
    gdf_hasil = gpd.sjoin(gdf_cuaca, gdf_lahan, how="left", predicate='within')
    
    # Kolom 'q_name19' dari data Anda berisi info 'Sawah', kita ganti namanya
    return gdf_hasil.rename(columns={'q_name19': 'jenis_lahan'})

In [ ]:
def terapkan_aturan_logika(df_hasil):
    """
    Menerapkan aturan bisnis untuk mengoreksi prediksi berdasarkan jenis lahan.
    """
    print("[5/5] Menerapkan aturan logika bisnis pada hasil prediksi...")
    rekomendasi_final = []
    
    for index, row in df_hasil.iterrows():
        prediksi_awal = row['prediksi_awal']
        jenis_lahan = row['jenis_lahan']
        
        prediksi_koreksi = prediksi_awal.copy()
        
        # Aturan untuk Sawah: Prioritaskan Padi
        if pd.notna(jenis_lahan) and 'Sawah' in jenis_lahan:
            padi_ada = any(p['nama'] == 'Padi' for p in prediksi_koreksi)
            
            rekomendasi_padi = {'nama': 'Padi', 'skor': 1.0, 'catatan': 'Sangat direkomendasikan karena lahan sawah'}
            
            if padi_ada:
                prediksi_koreksi = [p for p in prediksi_koreksi if p['nama'] != 'Padi']
            
            prediksi_koreksi.insert(0, rekomendasi_padi)
        
        # Anda bisa menambahkan aturan lain di sini, misal untuk 'Tegalan'
        # elif pd.notna(jenis_lahan) and 'Tegalan' in jenis_lahan:
        #     ... (logika untuk menaikkan skor jagung/ubi) ...
            
        rekomendasi_final.append(prediksi_koreksi[:3])
        
    df_hasil['rekomendasi_final'] = rekomendasi_final
    return df_hasil

In [ ]:
MODEL_FILE = '/home/noturminesv/projects/gis-ml/model/ModelClassifier.pkl'
CUACA_INPUT_FILE = '/home/noturminesv/projects/gis-ml/dataset/period3_psch_ht.csv'
PETA_LAHAN_FILE = '/home/noturminesv/projects/gis-ml/dataset/Peta_Lahan_Sawah_DIY_Final.geojson'

model, df_cuaca = muat_data_dan_model(MODEL_FILE, CUACA_INPUT_FILE)

if model and df_cuaca is not None:
    df_prediksi_awal = lakukan_prediksi_awal(model, df_cuaca)
    
    if df_prediksi_awal is not None:
        df_hasil_join = gabung_dengan_data_lahan(df_prediksi_awal, PETA_LAHAN_FILE)
        
        if df_hasil_join is not None:
            df_final = terapkan_aturan_logika(df_hasil_join)
            
            print("\n--- HASIL AKHIR REKOMENDASI YANG DISEMPURNAKAN ---")
            print(df_final[['LAT', 'LON', 'prediksi_awal', 'rekomendasi_final']].head())
            
            df_final = df_final.drop(columns=['geometry', 'index_right', 'objectid', 'wadmpr', 'wadmkk', 'luas_polyg', 'jenis_lahan'], errors='ignore')

            output_csv_file = "/home/noturminesv/projects/gis-ml/result/rekomendasi_final_dengan_lahan.csv"
            df_final.to_csv(output_csv_file, index=False)
            print(f"\nHasil lengkap disimpan di '{output_csv_file}'")

[1/5] Memuat model dan data input cuaca...
[2/5] Melakukan prediksi awal berdasarkan cuaca...
[3/5] Memuat peta tata guna lahan dari file lokal '../../dataset/Peta_Lahan_Sawah_DIY_Final.geojson'...
[4/5] Melakukan Spatial Join untuk identifikasi jenis lahan...
[5/5] Menerapkan aturan logika bisnis pada hasil prediksi...

--- HASIL AKHIR REKOMENDASI YANG DISEMPURNAKAN ---
    LAT     LON                                      prediksi_awal  \
0 -8.15  110.65  [{'nama': 'Pisang', 'skor': 0.8501818385025942...   
1 -8.15  110.70  [{'nama': 'Pisang', 'skor': 0.8869283260458743...   
2 -8.15  110.75  [{'nama': 'Pisang', 'skor': 0.8283721901847126...   
3 -8.10  110.45  [{'nama': 'Kacang Hijau', 'skor': 0.9398480412...   
4 -8.10  110.50  [{'nama': 'Kacang Hijau', 'skor': 0.7835063965...   

                                   rekomendasi_final  
0  [{'nama': 'Pisang', 'skor': 0.8501818385025942...  
1  [{'nama': 'Pisang', 'skor': 0.8869283260458743...  
2  [{'nama': 'Pisang', 'skor': 0.8283721